# Creating a Poisson Model to try and predict the score of a Premier League Game

Our target here is FTHG and FTAG

In [1]:
import pandas as pd
import numpy as np

In [2]:
url = "https://raw.githubusercontent.com/hankike/Premier_Prediction/refs/heads/main/data/cleaned_premier.csv"

df_premier = pd.read_csv(url)

In [3]:
# Sort original match dataframe chronologically
df_premier = df_premier.sort_values(
    ['Date', 'Time']
).reset_index(drop=True)

# Total goals in each match
df_premier['Total_Goals'] = (
    df_premier['FTHG'] + df_premier['FTAG']
)

# Cumulative league goals before each match
df_premier['League_Total_Goals'] = (
    df_premier['Total_Goals'].shift(1).cumsum()
)

# Number of previous matches
df_premier['League_Total_Matches'] = (
    df_premier['Total_Goals'].shift(1).expanding().count()
)

# Adding match ID
df_premier['Match_ID'] = range(len(df_premier))

# Average goals per team per match
df_premier['League_Avg_Goals'] = (
    df_premier['League_Total_Goals'] /
    (df_premier['League_Total_Matches'] * 2)
)

First I wish to update this data set in a way that makes every game into two rows, one for each team

In [4]:
# Home team
home_df = df_premier[
    ['Match_ID', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG']
].copy()

home_df = home_df.rename(columns={
    'HomeTeam': 'Team',
    'AwayTeam': 'Opponent',
    'FTHG': 'Goals_Scored',
    'FTAG': 'Goals_Allowed'
})

home_df['Home'] = 1

# Away team
away_df = df_premier[
    ['Match_ID', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG']
].copy()

away_df = away_df.rename(columns={
    'AwayTeam': 'Team',
    'HomeTeam': 'Opponent',
    'FTAG': 'Goals_Scored',
    'FTHG': 'Goals_Allowed'
})

away_df['Home'] = 0

# Combine
team_df = pd.concat([home_df, away_df], ignore_index=True)

# Convert the date
team_df['Date'] = pd.to_datetime(team_df['Date'], dayfirst=True)

# Sort
team_df = team_df.sort_values(['Date', 'Team']).reset_index(drop=True)

In [5]:
team_df = team_df.merge(
    df_premier[['Match_ID', 'League_Avg_Goals']],
    on='Match_ID',
    how='left'
)

In [6]:
# Create our points based on wins and draws
team_df['Points'] = np.select(
    [
        team_df['Goals_Scored'] > team_df['Goals_Allowed'],
        team_df['Goals_Scored'] == team_df['Goals_Allowed']
    ],
    [3, 1],
    default=0
)

# Count number of wins
team_df['Win'] = (
    team_df['Goals_Scored'] > team_df['Goals_Allowed']
).astype(int)

### Rolling Features

Next, I wish to capture how the teams are performing based on their attack, defense, and results.

To do this I am calculating how the teams have been performing recently using rolling features. This is an essential aspect of prediction modeling as you can never use future outcomes when finding your prediction values.

In [7]:
# Group by team
grouped = team_df.groupby('Team')

# Shift by 1 to exclude the current match
team_df['Avg_Goals_Scored_5'] = (
    grouped['Goals_Scored']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

team_df['Avg_Goals_Scored_10'] = (
    grouped['Goals_Scored']
    .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean())
)

team_df['Avg_Goals_Allowed_5'] = (
    grouped['Goals_Allowed']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

team_df['Avg_Goals_Allowed_10'] = (
    grouped['Goals_Allowed']
    .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean())
)

team_df['Points_Last_5'] = (
    grouped['Points']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
)

team_df['Wins_Last_5'] = (
    grouped['Win']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
)

### Offensive and Defensive Rankings

We wish to find metrics of how well a team is performing compared to league average

In [8]:
team_df['Attack_Strength'] = (
    team_df['Avg_Goals_Scored_10'] / team_df['League_Avg_Goals']
)

team_df['Defense_Strength'] = (
    team_df['Avg_Goals_Allowed_10'] / team_df['League_Avg_Goals']
)

In [9]:
team_df

,Match_ID,Date,Time,Team,Opponent,Goals_Scored,Goals_Allowed,Home,League_Avg_Goals,Points,Win,Avg_Goals_Scored_5,Avg_Goals_Scored_10,Avg_Goals_Allowed_5,Avg_Goals_Allowed_10,Points_Last_5,Wins_Last_5,Attack_Strength,Defense_Strength
0,768,2019-08-09,20:00,Liverpool,Norwich,4,1,1,1.480469,3,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,768,2019-08-09,20:00,Norwich,Liverpool,1,4,0,1.480469,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,846,2019-08-10,17:30,Aston Villa,Tottenham,1,3,0,1.473404,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,842,2019-08-10,15:00,Bournemouth,Sheffield United,1,1,1,1.475653,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,845,2019-08-10,15:00,Brighton,Watford,3,0,0,1.473373,3,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5315,2042,2026-05-24,16:00,Nott'm Forest,Bournemouth,1,1,1,1.427032,1,0,3.0,2.2,1.2,1.1,10.0,3.0,1.541661,0.770830
5316,2043,2026-05-24,16:00,Sunderland,Chelsea,2,1,1,1.426823,3,1,1.4,1.2,2.2,1.4,5.0,1.0,0.841029,0.981201
5317,2044,2026-05-24,16:00,Tottenham,Everton,1,0,1,1.426859,3,1,1.4,1.0,1.2,1.6,8.0,2.0,0.700840,1.121344
5318,2045,2026-05-24,16:00,West Ham,Leeds,3,0,1,1.426406,3,1,0.6,1.1,1.6,1.6,4.0,1.0,0.771169,1.121700
